# Tronco enrutador + mini-expertos LoRA por oficio

Un **maestro** (un LLM) etiqueta preguntas por oficio y escribe respuestas de ejemplo. Con eso se entrena:
1. un **router troncal** pequeño (tu «tálamo»), que decide el oficio de cada pregunta en milisegundos y sin gastar API;
2. un **mini-experto LoRA** por oficio (electricidad, agricultura, mecánica, redes) sobre un modelo base pequeño.

**Quién hace de maestro** (se elige solo):
- **DeepSeek**, si guardaste tu llave en *Secretos de Colab* como `DEEPSEEK_API_KEY` (no hace falta tocar el código).
- Si no hay llave, **un modelo pequeño de Hugging Face** (Qwen 2.5 de 1,5B), gratis en la GPU de Colab.
- Si tampoco se puede, **reglas por palabras clave** (solo para probar).

**Qué se mide:** el router contra una regresión logística (industria); y las respuestas del modelo base, de **un solo LoRA con todos los oficios** (el rival clave), de **router + LoRA por oficio** (tu idea) y del **oráculo** (el LoRA correcto siempre: el techo). Menos perplejidad = respuestas más parecidas a las del experto.

> *Entorno de ejecución → Cambiar tipo → GPU (T4)*. **Nunca escribas llaves en una celda.**

## 1. Preparar (≈ 3 min)

In [ ]:
import os, subprocess
from google.colab import userdata

try:  # DeepSeek es opcional: si no hay llave, el cuaderno usa un modelo de Hugging Face
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY") or ""
except Exception:
    os.environ["DEEPSEEK_API_KEY"] = ""
os.environ.setdefault("MAESTRO_HF", "Qwen/Qwen2.5-1.5B-Instruct")   # cámbialo si quieres otro maestro local

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt transformers peft accelerate sentence-transformers openai

from laboratorio.banco import expertos_lora as e
maestro = e.elegir_maestro()
print("Maestro:", maestro.nombre, "·", maestro.notas)

## 2. El maestro escribe preguntas y respuestas por oficio (≈ 10-30 min)

Con DeepSeek es rápido y barato; con el modelo de Hugging Face tarda más y las respuestas son más simples. Se guardan en Drive para no repetirlo.

In [ ]:
import json, random
from google.colab import drive
drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/Notbook/expertos"; os.makedirs(CARPETA, exist_ok=True)
ARCHIVO = f"{CARPETA}/pares_{maestro.nombre}.jsonl"

POR_DOMINIO, TANDA = 200, 20     # sube POR_DOMINIO si tienes créditos o tiempo
if os.path.exists(ARCHIVO):
    pares = [json.loads(l) for l in open(ARCHIVO)]
else:
    pares = []
    for d in e.DOMINIOS:
        while sum(p["dominio"] == d for p in pares) < POR_DOMINIO:
            pares += maestro.generar_pares(d, TANDA)
            print(d, sum(p["dominio"] == d for p in pares))
    with open(ARCHIVO, "w") as f:
        f.writelines(json.dumps(p, ensure_ascii=False) + "\n" for p in pares)
random.Random(0).shuffle(pares)
corte = int(0.8 * len(pares))
entreno, prueba = pares[:corte], pares[corte:]
print(len(entreno), "para entrenar,", len(prueba), "de prueba")

## 3. Router troncal contra la industria (≈ 2 min)

Las etiquetas de prueba son las del maestro. Ojo: si el maestro se equivoca, el router aprende sus errores; por eso conviene revisar a mano unas 50 preguntas.

In [ ]:
import numpy as np
r = e.comparar_routers([p["pregunta"] for p in entreno], [p["dominio"] for p in entreno],
                       [p["pregunta"] for p in prueba], [p["dominio"] for p in prueba])
print(f"Router troncal: {np.mean(r['router_troncal']):.1%} ± {np.std(r['router_troncal']):.1%}")
print(f"Regresión logística (industria): {r['logistica']:.1%}")

x, vec = e.embeddings([p["pregunta"] for p in entreno])
router = e.RouterTroncal(x.shape[1]).entrenar(x, [p["dominio"] for p in entreno])
for q, ds in e.MIXTAS:   # preguntas que mezclan oficios: ¿qué elige?
    print(q, "→", router.predecir(e.embeddings([q], vec)[0], top=2), "(esperado:", ds, ")")

## 4. Entrenar los mini-expertos LoRA (≈ 30-60 min)

Base: Qwen 2.5 de 0,5B (cabe y entrena rápido en una T4). Se entrenan 4 expertos (uno por oficio) y **un LoRA con todos los oficios juntos**, que es el rival que hay que vencer.

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
carpetas = {}
for d in e.DOMINIOS:
    carpetas[d] = e.entrenar_lora(BASE, [p for p in entreno if p["dominio"] == d], f"{CARPETA}/lora_{d}")
    print("listo", d)
carpetas["todos"] = e.entrenar_lora(BASE, entreno, f"{CARPETA}/lora_todos")
expertos = e.cargar_expertos(BASE, carpetas)

## 5. ¿Gana tu idea? (perplejidad de las respuestas de prueba; menos es mejor)

In [ ]:
dominio_router = {p["pregunta"]: d for p, d in zip(prueba, router.predecir(e.embeddings([p["pregunta"] for p in prueba], vec)[0]))}
casos = {
    "modelo base sin ajustar": lambda p: None,
    "un solo LoRA con todos los oficios (rival)": lambda p: "todos",
    "router + LoRA por oficio (tu idea)": lambda p: dominio_router[p["pregunta"]],
    "oráculo (LoRA correcto siempre: el techo)": lambda p: p["dominio"],
}
resultados = {nombre: e.perplejidad(expertos, prueba, elegir) for nombre, elegir in casos.items()}
for nombre, ppl in resultados.items():
    print(f"{nombre:45s} {ppl:8.2f}")
json.dump({"maestro": maestro.nombre, "router": r, "perplejidad": resultados}, open(f"{CARPETA}/resultados.json", "w"), indent=1)

**Cómo leerlo**
- Si «router + LoRA por oficio» queda **por debajo** del LoRA único, separar por oficio ayuda: tu idea funciona.
- Si queda **igual o encima**, un solo LoRA basta (es más simple de mantener).
- La distancia entre tu idea y el oráculo es lo que se pierde cuando el router se equivoca.

## 6. Probar el orquestador

In [ ]:
for q in ["¿Qué calibre de cable necesito para una bomba de riego de 2 HP?", "¿Cómo configuro una VLAN?"]:
    dominio, texto = e.responder(expertos, router, vec, q)
    print(f"[{dominio}] {q}\n{texto}\n")